In [1]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from statsmodels.tsa.stattools import acf
from scipy.io import arff
import pandas as pd
from hurst import compute_Hc

def Probability(Data, M, tau):
    z = np.zeros((len(Data) - (M - 1) * tau, M))

    for i in range(len(Data) - (M - 1) * tau):
        for j in range(M):
            z[i][j] = Data[i + j * tau] 

    e = np.argsort(z, axis = 1)
        
    pattern_count = {}
    for pattern in e:
        key = tuple(pattern)
        pattern_count[key] = pattern_count.get(key, 0) + 1
            
    total = sum(pattern_count.values())
    P = np.array([count / total for count in pattern_count.values()])
    return P

def Probability_U(P):
    return np.full(shape=len(P), fill_value=1/len(P))

def Probability_Const(P):
    p = np.zeros(len(P))
    p[0] = 1
    return p

def Entropy(P):
    P = P[P > 0]
    return -np.sum(P * np.log2(P))

def J(P1, P2):
    P3 = np.add(P1, P2) / 2
    return Entropy(P3) - 0.5 * (Entropy(P1) + Entropy(P2))

raw_data, meta = arff.loadarff('AbnormalHeartbeat_TRAIN.arff')
df = pd.DataFrame(raw_data)
feature_columns = [col for col in df.columns if col != 'target']
X = df[feature_columns].values.astype(np.float32)
Y = df['target'].values

In [306]:
Data = []
Data2 = []
m = 4
tau = 1
for i in range(len(X)):
    data = X[i].copy()
    prob = Probability(data, m, tau)
    prob_u = Probability_U(prob)
    prob_const = Probability_Const(prob)
    entropy = Entropy(prob) / Entropy(prob_u)
    complexity = entropy * (J(prob, prob_u) / J(prob_u, prob_const))
    hurst, c, result = compute_Hc(np.diff(np.log(np.abs(data) + 1e-8)), kind='change')
    Data.append([complexity, entropy, hurst, Y[i]])
    Data2.append([entropy - complexity, Y[i]])
Data2.sort()


In [307]:
cnt = 0
for i in range(203, -1, -1):
    print(Data2[i])
    if Data2[i][1] == b'Normal':
        cnt += 1
    if cnt == 6:
        break

[np.float64(0.9309952991363029), b'Abnormal']
[np.float64(0.9296620335675501), b'Abnormal']
[np.float64(0.9200511017173684), b'Abnormal']
[np.float64(0.8948649210403951), b'Abnormal']
[np.float64(0.8694922026901458), b'Abnormal']
[np.float64(0.8610967153346379), b'Abnormal']
[np.float64(0.859216567286867), b'Abnormal']
[np.float64(0.8503917953522467), b'Normal']
[np.float64(0.8373830355335481), b'Abnormal']
[np.float64(0.8363867030688931), b'Abnormal']
[np.float64(0.825279052164246), b'Abnormal']
[np.float64(0.7669930721452114), b'Abnormal']
[np.float64(0.7664356770884991), b'Abnormal']
[np.float64(0.750261345683092), b'Abnormal']
[np.float64(0.7412241345795623), b'Abnormal']
[np.float64(0.7309117814590664), b'Abnormal']
[np.float64(0.7207505424862446), b'Abnormal']
[np.float64(0.708758681036363), b'Abnormal']
[np.float64(0.7086973715229686), b'Abnormal']
[np.float64(0.7063247296344278), b'Abnormal']
[np.float64(0.705767736377672), b'Abnormal']
[np.float64(0.7035864891213464), b'Abnorm

In [302]:
a = 0
n = 0
Data2 = []
for i in range(204):
    if (Data[i][1] - Data[i][0] >= 0.461):
        Data2.append(Data[i])
        if (Data[i][3] == b'Abnormal'):
            a += 1
    else:
        if (Data[i][3] == b'Normal'):
            n += 1
print(a, n)

72 50


In [304]:
Data2.sort()
cnt = 0
for i in range(len(Data2)):
    print(Data2[i])
    if Data2[i][3] == b'Normal':
        cnt += 1
    if cnt == 2:
        break

[np.float64(0.037445190846581544), np.float64(0.9684404899828843), np.float64(0.18645292315055642), b'Abnormal']
[np.float64(0.039289554996589156), np.float64(0.9689515885641392), np.float64(0.192484262839908), b'Abnormal']
[np.float64(0.04231723912472848), np.float64(0.9623683408420969), np.float64(0.17570673160804964), b'Abnormal']
[np.float64(0.056041636501693734), np.float64(0.9509065575420887), np.float64(0.17052425454160208), b'Abnormal']
[np.float64(0.07152143795943754), np.float64(0.9410136406495834), np.float64(0.17333792774933457), b'Abnormal']
[np.float64(0.07739952358783142), np.float64(0.9384962389224694), np.float64(0.21488378293975266), b'Abnormal']
[np.float64(0.07786298775807525), np.float64(0.9370795550449422), np.float64(0.20776073078652077), b'Abnormal']
[np.float64(0.08202986063438136), np.float64(0.9324216559866281), np.float64(0.19615213681512458), b'Normal']
[np.float64(0.08379350115421581), np.float64(0.9211765366877639), np.float64(0.1910793657205302), b'Abnor

In [3]:
ans = []
for i1 in range(204):
    for i2 in range(204):
            NOR = 0
            ABNinNOR = 0
            ABN = 0
            NORinABN = 0
            for i4 in range(204):
                cnt = 0
                if (Data[i4][1] - Data[i4][0] <= Data[i1][1] - Data[i2][0]):
                    if (Y[i4] == b'Normal'):
                        NOR += 1
                    else:
                        ABNinNOR += 1
                else:
                    if (Y[i4] == b'Abnormal'):
                        ABN += 1
                    else:
                        NORinABN += 1
            ans.append([NOR, ABN, NORinABN, ABNinNOR])

In [4]:
mx = 0
for i in range(len(ans)):
    if (mx < ans[i][0] / 55 + ans[i][1] / 149):
        mx = ans[i][0] / 55 + ans[i][1] / 149
        print(ans[i][0], ans[i][1], ans[i][2], ans[i][3], ans[i][0] / 55 * 100, ans[i][1] / 149 * 100)

9 121 46 28 16.363636363636363 81.20805369127517
12 116 43 33 21.818181818181817 77.85234899328859
9 125 46 24 16.363636363636363 83.89261744966443
8 135 47 14 14.545454545454545 90.60402684563759
22 99 33 50 40.0 66.44295302013423
29 92 26 57 52.72727272727272 61.74496644295302
42 63 13 86 76.36363636363637 42.281879194630875
39 74 16 75 70.9090909090909 49.664429530201346
38 79 17 70 69.0909090909091 53.02013422818792
39 77 16 72 70.9090909090909 51.67785234899329
